# ⚖️ 台灣司法官律師國考 模型微調 (Kaggle GPU 專用極速版)
本 Notebook 專為 **Kaggle GPU (GPU T4 x2 或 P100)** 最佳化，提供完整的 Unsloth 4-bit 顯存優化與多源資料加權微調架構。

### ⚠️ 執行前 Kaggle 重要設定（必做！）：
1. **開啟 GPU 加速**：右側邊欄 `Settings` -> `Accelerator` 選擇 **`GPU T4 x2`**（或 P100）。
2. **開啟網路權限**：右側邊欄 `Settings` -> `Internet` 務必勾選 **`Internet on`**（下載模型與套件必備）。
3. **掛載訓練資料集**：右側邊欄點擊 `+ Add Input` 上傳您的 `law_exam_balanced_weighted_sft.jsonl`（或使用下方自動下載方式）。

In [ ]:
# 1. 安裝 Kaggle 專用 Unsloth 與依賴套件
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 指定主 GPU 進行極速微調

!pip install --upgrade --no-cache-dir "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo datasets trl peft bitsandbytes gdown

import torch
print('✅ GPU 設備狀態：', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '未偵測到 GPU，請在 Settings 開啟 GPU！')


In [ ]:
# 2. 自動偵測與載入微調資料集 (支援 Kaggle Input 或雲端直連)
import glob
from pathlib import Path

#@title 資料集路徑配置
# 自動搜尋 /kaggle/input 下的所有微調資料集
found_datasets = glob.glob("/kaggle/input/**/law_exam*.jsonl", recursive=True)

DATA_PATH = None
if found_datasets:
    # 優先選擇黃金加權庫
    for p in found_datasets:
        if "weighted" in p.lower():
            DATA_PATH = p
            break
    if not DATA_PATH:
        DATA_PATH = found_datasets[0]
    print(f'✅ 自動偵測到 Kaggle Input 資料集：{DATA_PATH}')
else:
    # 若未掛載 Input，提供備用示範路徑或透過 gdown 下載
    print('💡 提示：尚未在右側 + Add Input 掛載資料集。')
    print('您可直接點擊右側 + Add Input 上傳 law_exam_balanced_weighted_sft.jsonl，')
    print('或在下方填入 Google Drive 分享 ID 進行即時下載：')
    GDRIVE_FILE_ID = "" #@param {type:"string"}
    if GDRIVE_FILE_ID.strip():
        DATA_PATH = "/kaggle/working/dataset.jsonl"
        !gdown --id {GDRIVE_FILE_ID} -O {DATA_PATH}
    else:
        DATA_PATH = "/kaggle/working/sample_data.jsonl"
        # 建立測試樣本以確保 notebook 能直接試跑 (僅供語法測試)
        import json
        sample_item = {
            "id": "test_001",
            "source_type": "verified_user_handwritten_notes",
            "instruction": "請以 IRAC 架構解答刑法生命法益爭點。",
            "input": "醫師乙趁胎兒頭部未露出產道故意致其死亡，論以何罪？",
            "output": "採陣痛開始說（王皇玉、林山田），自然產以規則陣痛開始為人之始期，應成立刑法第271條普通殺人罪。"
        }
        with open(DATA_PATH, "w", encoding="utf-8") as f_s:
            for _ in range(50):
                f_s.write(json.dumps(sample_item, ensure_ascii=False) + "\n")
        print('=' * 75)
        print('⚠️⚠️⚠️ 重要警告：目前未偵測到真實資料集，正使用 50 筆重複測試樣本進行流程驗證！')
        print('⚠️ 請在 Kaggle 右側點擊 [+ Add Input] 掛載您的真實訓練集！')
        print('=' * 75)


In [ ]:
# 3. 載入基底開源模型 (Qwen2.5-7B-Instruct 4-bit) 與配置 LoRA
from unsloth import FastLanguageModel

max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

# 配置高效能 LoRA 轉接器
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print('✅ Qwen2.5-7B-Instruct 模型與 LoRA 介面載入成功！')


In [ ]:
# 4. 嚴格特徵綱要載入、動態加權過採樣與 Prompt 格式化
from datasets import load_dataset, Features, Value
from trl import SFTTrainer
from transformers import TrainingArguments
import random

# 100% 欄位相容防護 (徹底消除 CastError)
sft_features = Features({
    "id": Value("string"),
    "source_type": Value("string"),
    "instruction": Value("string"),
    "input": Value("string"),
    "output": Value("string"),
})

try:
    dataset = load_dataset('json', data_files=DATA_PATH, split='train', features=sft_features)
except Exception as e:
    print('Features load failed, falling back to standard loader:', e)
    dataset = load_dataset('json', data_files=DATA_PATH, split='train')

#@title 來源動態過採樣加權控制 (避免手寫筆記被稀釋)
ENABLE_DYNAMIC_WEIGHTING = True #@param {type:"boolean"}
NOTE_WEIGHT = 500 #@param {type:"integer"} 手寫筆記權重倍數
EXAM_WEIGHT = 50  #@param {type:"integer"} 司律真題權重倍數
COURT_WEIGHT = 15 #@param {type:"integer"} 憲法與裁判權重倍數

if ENABLE_DYNAMIC_WEIGHTING and "source_type" in dataset.column_names:
    sts = dataset["source_type"]
    idx_pool = []
    n_cnt, e_cnt, c_cnt, s_cnt = 0, 0, 0, 0
    for i, s in enumerate(sts):
        st = str(s).lower()
        if "handwritten" in st or "user_note" in st:
            idx_pool.extend([i] * NOTE_WEIGHT)
            n_cnt += NOTE_WEIGHT
        elif "exam" in st or "coursework" in st or "mcq" in st:
            idx_pool.extend([i] * EXAM_WEIGHT)
            e_cnt += EXAM_WEIGHT
        elif "constitutional" in st or "judgment" in st or "rule" in st or "theories" in st:
            idx_pool.extend([i] * COURT_WEIGHT)
            c_cnt += COURT_WEIGHT
        else:
            idx_pool.append(i)
            s_cnt += 1
    random.seed(3407)
    random.shuffle(idx_pool)
    dataset = dataset.select(idx_pool)
    total = len(dataset)
    print(f'✅ 加權平衡完成！加權後訓練總樣本數：{total:,} 筆')
    print(f'   - 📝 手寫筆記佔比：{n_cnt/total*100:.2f}% ({n_cnt:,} 次曝光)')
    print(f'   - 🎓 司律真題/講義佔比：{e_cnt/total*100:.2f}% ({e_cnt:,} 次曝光)')
    print(f'   - 🏛️ 憲法/裁判規準佔比：{c_cnt/total*100:.2f}% ({c_cnt:,} 次曝光)')
    print(f'   - 📚 基礎模板語料佔比：{s_cnt/total*100:.2f}% ({s_cnt:,} 次曝光)')

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = (
            f"<|im_start|>system\n你是一位具備多年命題閱卷經驗之台灣司法官律師考試閱卷委員與法學權威。<|im_end|>\n"
            f"<|im_start|>user\n{instruction}\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
        texts.append(text)
    return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'成功載入並格式化 {len(dataset):,} 筆微調樣本！')


In [ ]:
# 5. 啟動 SFT 訓練迴圈 (Kaggle GPU 極速加速)
from unsloth import is_bfloat16_supported

#@title 設定訓練步數 (Training Steps)
max_training_steps = 200 #@param {type:"integer"}

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = max_training_steps,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/kaggle/working/outputs",
    ),
)

print('🚀 開始訓練...')
trainer_stats = trainer.train()
print('🎉 訓練完成！統計資訊：', trainer_stats)


In [ ]:
# 6. 儲存 LoRA 權重並打包為 ZIP 供一鍵下載
import shutil

SAVE_DIR = "/kaggle/working/qwen_law_evaluator_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'✅ LoRA 權重已儲存至：{SAVE_DIR}')

# 打包為 zip 檔，方便從 Kaggle 右側 Output 面板直接點擊 Download
zip_path = "/kaggle/working/qwen_law_evaluator_lora.zip"
shutil.make_archive("/kaggle/working/qwen_law_evaluator_lora", 'zip', SAVE_DIR)
print(f'📦 已打包下載檔：{zip_path} (您可直接在右側 Output 面板點擊下載！)')

# (選填) 若有 Hugging Face 帳號，亦可直接 1 鍵推送到雲端 Hub：
# HUGGINGFACE_TOKEN = "hf_xxxx" #@param {type:"string"}
# HF_REPO_NAME = "your_name/qwen_law_evaluator_lora" #@param {type:"string"}
# if HUGGINGFACE_TOKEN.strip():
#     model.push_to_hub(HF_REPO_NAME, token=HUGGINGFACE_TOKEN)
#     tokenizer.push_to_hub(HF_REPO_NAME, token=HUGGINGFACE_TOKEN)
#     print(f'🚀 已成功推送到 Hugging Face：https://huggingface.co/{HF_REPO_NAME}')
